In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

sample = "Word embeddings are dense vector representations of words."
tokenized_corpus = word_tokenize(sample.lower())

skipgram_model = Word2Vec(sentences=[tokenized_corpus],
                          vector_size=100,
                          window=5,
                          sg=1,
                          min_count=1,
                          workers=4)

# Training
skipgram_model.train([tokenized_corpus], total_examples=1, epochs=10)
skipgram_model.save("skipgram_model.model")
loaded_model = Word2Vec.load("skipgram_model.model")
vector_representation = loaded_model.wv['word']
print("Vector representation of 'word':", vector_representation)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Vector representation of 'word': [-9.5800208e-03  8.9437785e-03  4.1664648e-03  9.2367809e-03
  6.6457358e-03  2.9233587e-03  9.8055992e-03 -4.4231843e-03
 -6.8048164e-03  4.2256550e-03  3.7299085e-03 -5.6668529e-03
  9.7035142e-03 -3.5551414e-03  9.5499391e-03  8.3657773e-04
 -6.3355025e-03 -1.9741615e-03 -7.3781307e-03 -2.9811086e-03
  1.0425397e-03  9.4814906e-03  9.3598543e-03 -6.5986011e-03
  3.4773252e-03  2.2767992e-03 -2.4910474e-03 -9.2290826e-03
  1.0267317e-03 -8.1645092e-03  6.3240929e-03 -5.8001447e-03
  5.5353874e-03  9.8330071e-03 -1.5987856e-04  4.5296676e-03
 -1.8086446e-03  7.3613892e-03  3.9419360e-03 -9.0095028e-03
 -2.3953868e-03  3.6261671e-03 -1.0080514e-04 -1.2024897e-03
 -1.0558038e-03 -1.6681013e-03  6.0541567e-04  4.1633579e-03
 -4.2531900e-03 -3.8336846e-03 -5.0755290e-05  2.6549282e-04
 -1.7014991e-04 -4.7843382e-03  4.3120929e-03 -2.1710952e-03
  2.1056964e-03  6.6702347e-04  5.9686624e-03 -6.8418151e-03
 -6.8183104e-03 -4.4762432e-03  9.4359247e-03 -1.593

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define CBOW model
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, context):
        # Sum along the context word dimension (dim=0) to get a single context embedding
        context_embeds = self.embeddings(context).sum(dim=0)
        output = self.linear(context_embeds)
        return output

context_size = 2
raw_text = "word embeddings are awesome and truly powerful features"
tokens = raw_text.split()
vocab = set(tokens)
word_to_index = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(context_size, len(tokens) - context_size):
    context = [word_to_index[word] for word in tokens[i - context_size:i] + tokens[i + 1:i + context_size + 1]]
    target = word_to_index[tokens[i]]
    data.append((torch.tensor(context), torch.tensor(target)))

# Hyperparameters
vocab_size = len(vocab)
embed_size = 10
learning_rate = 0.01
epochs = 100

# Initialize CBOW model
cbow_model = CBOWModel(vocab_size, embed_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(cbow_model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    total_loss = 0
    if not data: # Handle case where data might still be empty for very short texts
        print(f"Epoch {epoch + 1}, No data to train on. Total loss: 0")
        continue
    for context, target in data:
        optimizer.zero_grad()
        output = cbow_model(context)
        loss = criterion(output.unsqueeze(0), target.unsqueeze(0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss}")

# Example usage
word_to_lookup = "embeddings"
# Ensure word_to_lookup is in the vocabulary before attempting to get its index
if word_to_lookup in word_to_index:
    word_index = word_to_index[word_to_lookup]
    embedding = cbow_model.embeddings(torch.tensor([word_index]))
    print(f"Embedding for '{word_to_lookup}': {embedding.detach().numpy()}")
else:
    print(f"'{word_to_lookup}' not found in vocabulary.")

Epoch 1, Loss: 8.737441718578339
Epoch 2, Loss: 7.523172616958618
Epoch 3, Loss: 6.501645267009735
Epoch 4, Loss: 5.65723180770874
Epoch 5, Loss: 4.9684334099292755
Epoch 6, Loss: 4.407643049955368
Epoch 7, Loss: 3.9473286867141724
Epoch 8, Loss: 3.5644364655017853
Epoch 9, Loss: 3.2412444055080414
Epoch 10, Loss: 2.96456977725029
Epoch 11, Loss: 2.7247003316879272
Epoch 12, Loss: 2.514478772878647
Epoch 13, Loss: 2.3285887837409973
Epoch 14, Loss: 2.1630289405584335
Epoch 15, Loss: 2.0147309005260468
Epoch 16, Loss: 1.8812875300645828
Epoch 17, Loss: 1.760767638683319
Epoch 18, Loss: 1.651586875319481
Epoch 19, Loss: 1.5524182468652725
Epoch 20, Loss: 1.462131753563881
Epoch 21, Loss: 1.3797544091939926
Epoch 22, Loss: 1.3044375330209732
Epoch 23, Loss: 1.2354397177696228
Epoch 24, Loss: 1.1721065491437912
Epoch 25, Loss: 1.1138610392808914
Epoch 26, Loss: 1.0601920261979103
Epoch 27, Loss: 1.0106466561555862
Epoch 28, Loss: 0.9648233577609062
Epoch 29, Loss: 0.9223641827702522
Epoch 

https://www.geeksforgeeks.org/nlp/word-embeddings-in-nlp/


In [ ]:
from gensim.models import KeyedVectors
from gensim.downloader import load

glove_model = load('glove-wiki-gigaword-50')
word_pairs = [('learn', 'learning'), ('india', 'indian'), ('fame', 'famous')]

# Compute similarity for each pair of words
for pair in word_pairs:
    similarity = glove_model.similarity(pair[0], pair[1])
    print(f"Similarity between '{pair[0]}' and '{pair[1]}' using GloVe: {similarity:.3f}")

[==================================================] 100.0% 66.0/66.0MB downloaded
Similarity between 'learn' and 'learning' using GloVe: 0.802
Similarity between 'india' and 'indian' using GloVe: 0.865
Similarity between 'fame' and 'famous' using GloVe: 0.589


In [ ]:
import gensim.downloader as api
fasttext_model = api.load("fasttext-wiki-news-subwords-300") ## Load the pre-trained fastText model

word_pairs = [('learn', 'learning'), ('india', 'indian'), ('fame', 'famous')]

# Compute similarity for each pair of words
for pair in word_pairs:
    similarity = fasttext_model.similarity(pair[0], pair[1])
    print(f"Similarity between '{pair[0]}' and '{pair[1]}' using FastText: {similarity:.3f}")

[==================================================] 100.0% 958.5/958.4MB downloaded
Similarity between 'learn' and 'learning' using FastText: 0.642
Similarity between 'india' and 'indian' using FastText: 0.708
Similarity between 'fame' and 'famous' using FastText: 0.519


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

word_pairs = [('learn', 'learning'), ('india', 'indian'), ('fame', 'famous')]

# Compute similarity for each pair of words
for pair in word_pairs:
    tokens = tokenizer(pair, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**tokens)

    # Extract embeddings for the [CLS] token
    cls_embedding = outputs.last_hidden_state[:, 0, :]

    similarity = torch.nn.functional.cosine_similarity(cls_embedding[0], cls_embedding[1], dim=0)
    print(f"Similarity between '{pair[0]}' and '{pair[1]}' using BERT: {similarity:.3f}")

https://www.geeksforgeeks.org/nlp/nltk-tutorial/